# 🔬 RetinaSight — Kaggle GPU Model Training Pipeline
**Smart India Hackathon 2026** | **Problem Statement ID:** 26038 (MathWorks) | **Team OnFocus**

This notebook trains the deep learning Diabetic Retinopathy classifier using the **APTOS 2019 Blindness Detection** dataset on Kaggle GPUs (Tesla T4 or P100).

### 🏆 Architectural Highlights (APTOS 1st & 5th Place Solutions):
1. **Backbone:** TorchVision ResNet-50 with **Generalized Mean Pooling (GeM)**.
2. **Loss Formulation:** Continuous ordinal regression via `SmoothL1Loss` to penalize cross-severity misclassification.
3. **Preprocessing:** Ben Graham's circular aperture cropping (`crop_image_from_gray`) + green-channel CLAHE.
4. **Threshold Optimization:** `OptimizedRounder` via Nelder-Mead simplex search to maximize Quadratic Weighted Kappa (QWK).
5. **Production Export:** Automatically exports 5-class linear head to `retinasight_resnet50.onnx` and `retinasight_resnet50.pth` directly into `/kaggle/working/` for immediate download.


In [ ]:
"""
==================================================================================
RetinaSight — Kaggle GPU Training & Calibration Pipeline
Smart India Hackathon 2026 | PS ID 26038 (MathWorks) | Team OnFocus
==================================================================================

Winning APTOS 2019 Architectural Design:
1. Backbone: Pretrained ResNet-50 with Generalized Mean Pooling (GeM)
2. Loss: Smooth L1 Loss (Huber regression on continuous ordinal DR severity)
3. Optimization: AdamW + Cosine Annealing + Mixed Precision (fp16 AMP)
4. Evaluation: Quadratic Weighted Kappa (QWK) with Nelder-Mead OptimizedRounder
5. Export: Direct deployment to 'retinasight_resnet50.onnx' and 'retinasight_resnet50.pth'
==================================================================================
"""

import os
from pathlib import Path
import sys
import time

import cv2
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import classification_report, cohen_kappa_score
from sklearn.model_selection import StratifiedKFold, train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
import torchvision.transforms as transforms

# ----------------------------------------------------------------------------------
# 1. Hardware & Environment Setup
# ----------------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[ENVIRONMENT] Computing Device: {device}")
if torch.cuda.is_available():
	print(f"[GPU] Device Name: {torch.cuda.get_device_name(0)}")
	print(f"[GPU] VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Kaggle Dataset Paths
KAGGLE_INPUT_DIR = Path("/kaggle/input/aptos2019-blindness-detection")
LOCAL_INPUT_DIR = Path("D:/SIH2026/Datasets/aptos2019")

DATA_DIR = KAGGLE_INPUT_DIR if KAGGLE_INPUT_DIR.exists() else LOCAL_INPUT_DIR
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 256
BATCH_SIZE = 32 if torch.cuda.is_available() else 8
NUM_EPOCHS = 12
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4

ICDR_CLASSES = {
	0: "No DR",
	1: "Mild",
	2: "Moderate",
	3: "Severe",
	4: "Proliferative DR",
}


# ----------------------------------------------------------------------------------
# 2. Ben's Preprocessing (Circular FOV Crop)
# ----------------------------------------------------------------------------------
def crop_image_from_gray(img: np.ndarray, tol: int = 7) -> np.ndarray:
	"""Remove uninformative black background margins around the retinal aperture."""
	if img.ndim == 2:
		mask = img > tol
		return img[np.ix_(mask.any(1), mask.any(0))]
	elif img.ndim == 3:
		gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
		mask = gray_img > tol
		check_shape = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))].shape[0]
		if check_shape == 0:
			return img
		else:
			img1 = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))]
			img2 = img[:, :, 1][np.ix_(mask.any(1), mask.any(0))]
			img3 = img[:, :, 2][np.ix_(mask.any(1), mask.any(0))]
			return np.stack([img1, img2, img3], axis=-1)
	return img


def preprocess_fundus(img_bgr: np.ndarray, target_size: int = IMAGE_SIZE) -> np.ndarray:
	"""Apply circular aperture crop, CLAHE on green channel, and standardization."""
	cropped = crop_image_from_gray(img_bgr)
	resized = cv2.resize(cropped, (target_size, target_size), interpolation=cv2.INTER_AREA)

	# CLAHE on green channel for maximum hemoglobin contrast
	b, g, r = cv2.split(resized)
	clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
	g_clahe = clahe.apply(g)
	enhanced_bgr = cv2.merge([b, g_clahe, r])
	enhanced_rgb = cv2.cvtColor(enhanced_bgr, cv2.COLOR_BGR2RGB)
	return enhanced_rgb


# ----------------------------------------------------------------------------------
# 3. Dataset & PyTorch DataLoader
# ----------------------------------------------------------------------------------
class AptosKaggleDataset(Dataset):
	def __init__(self, df: pd.DataFrame, image_dir: Path, is_train: bool = True):
		self.df = df.reset_index(drop=True)
		self.image_dir = image_dir
		self.is_train = is_train

		# Build fast lookup map
		self.id_map = {}
		for ext in ("*.png", "*.jpg"):
			for p in self.image_dir.rglob(ext):
				self.id_map[p.stem] = p

		# Base transforms
		self.norm = transforms.Compose([
			transforms.ToPILImage(),
			transforms.RandomHorizontalFlip(p=0.5 if is_train else 0.0),
			transforms.RandomVerticalFlip(p=0.5 if is_train else 0.0),
			transforms.RandomRotation(degrees=180 if is_train else 0),
			transforms.ToTensor(),
			transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
		])

	def __len__(self) -> int:
		return len(self.df)

	def __getitem__(self, idx: int):
		row = self.df.iloc[idx]
		code = str(row["id_code"])
		label = float(row["diagnosis"])

		img_path = self.id_map.get(code)
		if img_path and img_path.exists():
			img_bgr = cv2.imread(str(img_path))
		else:
			img_bgr = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)

		if img_bgr is None:
			img_bgr = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)

		prep_rgb = preprocess_fundus(img_bgr, target_size=IMAGE_SIZE)
		tensor = self.norm(prep_rgb)
		return tensor, torch.tensor(label, dtype=torch.float32)


# ----------------------------------------------------------------------------------
# 4. Model Architecture: ResNet-50 + GeM Pooling + Ordinal Regression
# ----------------------------------------------------------------------------------
class GeM(nn.Module):
	"""Generalized Mean Pooling (Kaggle 1st Place Solution)."""
	def __init__(self, p: float = 3.0, eps: float = 1e-6):
		super().__init__()
		self.p = nn.Parameter(torch.ones(1) * p)
		self.eps = eps

	def forward(self, x: torch.Tensor) -> torch.Tensor:
		return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1.0 / self.p)


class RetinaSightNet(nn.Module):
	def __init__(self, pretrained: bool = True):
		super().__init__()
		backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
		self.conv1 = backbone.conv1
		self.bn1 = backbone.bn1
		self.relu = backbone.relu
		self.maxpool = backbone.maxpool

		self.layer1 = backbone.layer1
		self.layer2 = backbone.layer2
		self.layer3 = backbone.layer3
		self.layer4 = backbone.layer4

		self.gem = GeM(p=3.0)
		self.fc = nn.Linear(2048, 1)  # Continuous regression output for SmoothL1Loss

	def forward(self, x: torch.Tensor) -> torch.Tensor:
		x = self.conv1(x)
		x = self.bn1(x)
		x = self.relu(x)
		x = self.maxpool(x)

		x = self.layer1(x)
		x = self.layer2(x)
		x = self.layer3(x)
		x = self.layer4(x)

		x = self.gem(x)
		x = x.flatten(1)
		out = self.fc(x)
		return out.squeeze(-1)


# ----------------------------------------------------------------------------------
# 5. Nelder-Mead OptimizedRounder for QWK Maximization
# ----------------------------------------------------------------------------------
class OptimizedRounder:
	def __init__(self, initial_coef=[0.5, 1.5, 2.5, 3.5]):
		self.coef_ = list(initial_coef)

	def _loss(self, coef, X, y):
		res = np.zeros(len(X), dtype=int)
		for i, pred in enumerate(X):
			if pred < coef[0]:
				res[i] = 0
			elif pred < coef[1]:
				res[i] = 1
			elif pred < coef[2]:
				res[i] = 2
			elif pred < coef[3]:
				res[i] = 3
			else:
				res[i] = 4
		return -cohen_kappa_score(y, res, weights="quadratic")

	def fit(self, X, y):
		res = minimize(self._loss, self.coef_, args=(X, y), method="Nelder-Mead")
		self.coef_ = res.x

	def predict(self, X):
		res = np.zeros(len(X), dtype=int)
		for i, pred in enumerate(X):
			if pred < self.coef_[0]:
				res[i] = 0
			elif pred < self.coef_[1]:
				res[i] = 1
			elif pred < self.coef_[2]:
				res[i] = 2
			elif pred < self.coef_[3]:
				res[i] = 3
			else:
				res[i] = 4
		return res


# ----------------------------------------------------------------------------------
# 6. Training Pipeline
# ----------------------------------------------------------------------------------
def train_retinasight():
	print("=" * 80)
	print(" RETINASIGHT — KAGGLE GPU ACCELERATED MODEL TRAINING PIPELINE")
	print("=" * 80)

	csv_path = DATA_DIR / "train.csv"
	if not csv_path.exists():
		print(f"[ERROR] Could not find train.csv at: {csv_path}")
		return

	df = pd.read_csv(csv_path)
	print(f"[DATASET] Loaded {len(df)} total entries from {csv_path}")
	print(df["diagnosis"].value_counts().sort_index())

	# Stratified 80/20 train/validation split
	train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["diagnosis"])
	print(f"[SPLIT] Training Set: {len(train_df)} | Validation Set: {len(val_df)}")

	train_dataset = AptosKaggleDataset(train_df, DATA_DIR, is_train=True)
	val_dataset = AptosKaggleDataset(val_df, DATA_DIR, is_train=False)

	train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
	val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

	# Initialize Model, Loss, Optimizer & Scheduler
	model = RetinaSightNet(pretrained=True).to(device)
	criterion = nn.SmoothL1Loss()
	optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
	scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
	scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

	best_qwk = -1.0
	best_weights = None

	for epoch in range(1, NUM_EPOCHS + 1):
		t0 = time.time()
		model.train()
		train_loss = 0.0

		for images, labels in train_loader:
			images = images.to(device, non_blocking=True)
			labels = labels.to(device, non_blocking=True)

			optimizer.zero_grad()
			with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
				preds = model(images)
				loss = criterion(preds, labels)

			scaler.scale(loss).backward()
			scaler.step(optimizer)
			scaler.update()

			train_loss += loss.item() * len(labels)

		scheduler.step()
		train_loss /= len(train_dataset)

		# Validation Evaluation
		model.eval()
		val_preds_cont, val_targets = [], []
		with torch.no_grad():
			for images, labels in val_loader:
				images = images.to(device, non_blocking=True)
				with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
					preds = model(images)
				val_preds_cont.extend(preds.cpu().numpy().tolist())
				val_targets.extend(labels.numpy().tolist())

		val_preds_cont = np.array(val_preds_cont)
		val_targets = np.array(val_targets)

		# Standard rounding [0.5, 1.5, 2.5, 3.5]
		default_rounder = OptimizedRounder([0.5, 1.5, 2.5, 3.5])
		default_preds = default_rounder.predict(val_preds_cont)
		default_qwk = cohen_kappa_score(val_targets, default_preds, weights="quadratic")

		# Optimized rounding
		opt_rounder = OptimizedRounder([0.5, 1.5, 2.5, 3.5])
		opt_rounder.fit(val_preds_cont, val_targets)
		opt_preds = opt_rounder.predict(val_preds_cont)
		opt_qwk = cohen_kappa_score(val_targets, opt_preds, weights="quadratic")

		epoch_time = time.time() - t0
		print(
			f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] ({epoch_time:.1f}s) | "
			f"Train Loss: {train_loss:.4f} | "
			f"Val QWK (Default): {default_qwk:.4f} | "
			f"Val QWK (Optimized): {opt_qwk:.4f} | "
			f"Thresholds: {[round(c, 2) for c in opt_rounder.coef_]}"
		)

		if opt_qwk > best_qwk:
			best_qwk = opt_qwk
			best_weights = model.state_dict()
			torch.save(best_weights, str(OUTPUT_DIR / "best_retinasight_model.pth"))
			print(f"  >>> Best Model Checkpoint Saved! (QWK: {best_qwk:.4f})")

	print("\n" + "=" * 80)
	print(f" TRAINING COMPLETE | Peak Validation QWK: {best_qwk:.4f}")
	print("=" * 80)

	# ----------------------------------------------------------------------------------
	# 7. Production Model Packaging (5-Class Linear Head for ONNX & Grad-CAM)
	# ----------------------------------------------------------------------------------
	print("\n[PACKAGING] Converting trained continuous backbone to 5-Class ONNX format...")
	production_model = models.resnet50(weights=None)
	# Transfer convolutional weights
	prod_dict = production_model.state_dict()
	for k, v in best_weights.items():
		if not k.startswith("fc.") and k in prod_dict:
			prod_dict[k] = v
	production_model.load_state_dict(prod_dict)

	# Construct linear 5-class calibrated head
	# Logit_k = 2k(w^T x + b) - k^2
	w_reg = best_weights["fc.weight"].cpu().numpy().flatten()
	b_reg = best_weights["fc.bias"].cpu().numpy().flatten()[0]

	W_5class = np.zeros((5, 2048), dtype=np.float32)
	b_5class = np.zeros(5, dtype=np.float32)
	for k in range(5):
		W_5class[k] = 2.0 * k * w_reg
		b_5class[k] = 2.0 * k * b_reg - (k ** 2)

	production_model.fc = nn.Linear(2048, 5)
	production_model.fc.weight.data = torch.from_numpy(W_5class).float()
	production_model.fc.bias.data = torch.from_numpy(b_5class).float()
	production_model.eval()

	# Save PyTorch production checkpoint
	prod_pth = OUTPUT_DIR / "retinasight_resnet50.pth"
	torch.save(production_model.state_dict(), str(prod_pth))
	print(f"[SAVED] Production PyTorch Model: {prod_pth}")

	# Export production ONNX graph
	prod_onnx = OUTPUT_DIR / "retinasight_resnet50.onnx"
	dummy_input = torch.randn(1, 3, 256, 256, dtype=torch.float32)
	torch.onnx.export(
		production_model,
		dummy_input,
		str(prod_onnx),
		export_params=True,
		opset_version=14,
		do_constant_folding=True,
		input_names=["input"],
		output_names=["logits"],
		dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
		dynamo=False,
	)
	print(f"[SAVED] Production ONNX Graph: {prod_onnx} ({prod_onnx.stat().st_size / (1024**2):.2f} MB)")
	print("\n[READY] Download 'retinasight_resnet50.onnx' and 'retinasight_resnet50.pth' from Kaggle output!")


if __name__ == "__main__":
	train_retinasight()
